# Max-mix many-body Lyapunov A100 pilot

This notebook runs a simple completion-resumable campaign for the hard chiral wall. Starting from the maximally mixed state, the active covariance occupations and the realized Born record weight reconstruct the leading many-body squared singular values through $\log Z=N_{\rm eff}\log 2+\omega$ and $\ell_i=\log\sigma_i^2$. Code runs locally under `/content`; Drive stores only verified task products.

## 1. Mount Drive and set the complete campaign configuration

Normally only `REPORT_ONLY`, `MAX_NEW_TASKS`, and `RUN_ANALYSIS` should be changed. The scientific dictionary is locked and validated by the runner.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

NOTEBOOK_BUILD = 'lyapunov-v2-a100-cap-roundoff-fix'
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/04_maxmix_manybody_lyapunov_pilot')
LOCAL_BUNDLE_DIR = Path('/content/04_maxmix_manybody_lyapunov_pilot')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/maxmix_manybody_lyapunov_nx20_ny20to40_s100_gpu_v2')
SCRATCH_ROOT = Path('/content/maxmix_manybody_lyapunov_scratch')
REPORT_ONLY = False
MAX_NEW_TASKS = None  # Set to 1 for a single Ny=40 timing task.
RUN_ANALYSIS = False  # Enable only after the report shows 160/160 tasks.

CONFIG = {
    'sampling_revision': 'maxmix_manybody_lyapunov_nx20_ny20to40_s100_gpu_v2',
    'root_seed': 2026090304,
    'Nx': 20,
    'Ny_values': [20, 22, 24, 26, 28, 30, 36, 40],
    'samples_per_Ny': 100,
    'shard_size': 5,
    'cycles_rule': '2*Ny',
    'device': 'cuda:0',
    'dtype': 'complex128',
    'protocol': {
        'DW': True,
        'domain_wall_interval': [5, 15],
        'dw_truncation': True,
        'meas_slab_only': True,
        'nshell': 1,
        'alpha_1': 1.0,
        'alpha_2': 30.0,
        'filling_frac': 0.5,
        'trial_orbitals': 'X',
        'init_mode': 'maxmix',
        'sequence': 'raster_y',
        'perfect_correction': True,
        'postselect': False,
        'postselect_probability': 0.0,
        'n_a': 0.5,
        'state_representation': 'covariance',
        'backend': 'local',
        'triv_region_local_mode': False,
    },
    'observer': {
        'spectrum_cycle_stride': 4,
        'extra_spectrum_cycles': ['Ny', '3*Ny/2', '2*Ny'],
        'soft_mode_count': 16,
        'leading_level_count': 64,
        'cap_tolerance': 1e-9,
    },
    'analysis': {
        'bootstrap_replicates': 2000,
        'bootstrap_seed': 2026090305,
        'relative_shift_threshold': 0.1,
        'sample_prefixes': [25, 50, 75, 100],
    },
}
print(f'Notebook build: {NOTEBOOK_BUILD}')
print(f'Bundle: {BUNDLE_DRIVE_DIR}')
print(f'Output: {OUTPUT_ROOT}')
print('Workload: 8 sizes, 160 five-trajectory tasks, 800 trajectories')

## 2. Stage locally and run or resume

The runner verifies every existing NPZ/completion pair, prints setup and task-start messages, shows an outer task bar, and surfaces the canonical engine's inner cycle bar. Valid completed tasks are skipped; only an interrupted five-trajectory task repeats.

In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

def run_streaming_child(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            decoded = decoder.decode(chunk)
            if decoded:
                sys.stdout.write(decoded)
                sys.stdout.flush()
        tail = decoder.decode(b'', final=True)
        if tail:
            sys.stdout.write(tail)
            sys.stdout.flush()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    returncode = process.wait()
    if returncode:
        print(
            f'[runner failed] child exit code={returncode}; the full child output is above',
            flush=True,
        )
        raise subprocess.CalledProcessError(returncode, command)

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/maxmix_manybody_lyapunov_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [
    sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
    '--config', str(config_path),
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_TASKS is not None:
    command.extend(['--max-new-tasks', str(int(MAX_NEW_TASKS))])
print('[launch]', ' '.join(command), flush=True)
run_streaming_child(command)
print('[notebook] runner exited successfully', flush=True)

## 3. Optional completed-campaign analysis

The analysis refuses an incomplete campaign by default and writes the finite-size tables and double-column PDF/PNG under the campaign output folder.

In [ ]:
if RUN_ANALYSIS:
    analysis_command = [
        sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'analyze_campaign.py'),
        '--config', str(config_path),
        '--results-root', str(OUTPUT_ROOT),
        '--output-root', str(OUTPUT_ROOT / 'analysis'),
    ]
    print('[analysis launch]', ' '.join(analysis_command), flush=True)
    run_streaming_child(analysis_command)
else:
    print('[analysis] skipped; set RUN_ANALYSIS=True after 160/160 tasks')

## 4. Release the runtime

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')